<a href="https://colab.research.google.com/github/faxml/Rekomendasi-penyiraman-cabai-menggunakan-sensor-DHT22-dan-kelembapan-tanah-berbasis-IOT/blob/main/model_penyiraman_cabai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MODEL TRAINING DATASET SERTA KODE PROGRAM
google colab ini berisi hasil training dataset menggunakan algoritma decision tree dan kode program yang digunakan pada sistem ini

# Model Rekomendasi Penyiraman Cabai (3 Kelas) untuk ESP32
Decision tree dilatih pada dataset *Soil Moisture, Air Temperature and Humidity, and Water Motor On/Off Monitor Data*.
Label 3 kelas dibentuk dari aturan berbasis kelembapan tanah dan suhu (lihat langkah 3).

| Kelas | Teks di LCD/Blynk |
|---|---|
| 0 | tidak perlu siram |
| 1 | Siram Ringan |
| 2 | Wajib Siram! |

In [ ]:
!pip install micromlgen

import pandas as pd
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix
from micromlgen import port   # pip install micromlgen

## 1. Muat dataset
File CSV harus berada satu folder dengan notebook ini.

In [ ]:
df = pd.read_csv("dataset_3_kelas_aman.csv")
print(df.shape)
# df.head()
pd.concat([df.head(5), df.tail(5)])

(3000, 6)


,Soil Moisture,Temperature,Air Humidity,Pump Data,soil_pct,Pump_Class
0,683.802906,29.184908,71.789699,0,55.092127,0
1,408.571567,33.707205,77.977391,1,14.032249,2
2,659.092074,24.760311,60.776282,1,51.405687,0
3,842.929764,32.738515,59.323543,0,78.831173,0
4,414.199320,25.692744,66.624914,1,14.871815,1
2995,355.566758,22.070900,71.290541,1,6.124825,1
2996,443.521135,36.829497,51.394630,1,19.246136,2
2997,935.440437,18.956660,77.663509,0,92.632206,0
2998,850.602693,22.342141,46.442113,0,79.975844,0
2999,752.875788,31.893127,40.656068,0,65.396637,0


## 2. Normalisasi kelembapan tanah ke persen (0 = kering, 100 = basah)
Nilai `lo` dan `hi` dipakai hanya untuk data latih. Di ESP32, persen dihitung dari kalibrasi ADC sensor (SOIL_DRY dan SOIL_WET).

In [ ]:
lo, hi = df["Soil Moisture"].min(), df["Soil Moisture"].max()
df["soil_pct"] = (df["Soil Moisture"] - lo) / (hi - lo) * 100
print("lo =", lo, "| hi =", hi)

lo = 314.5110162 | hi = 984.82801


## 3. Aturan pelabelan 3 kelas
Ambang 40%, 60%, 30 C, dan 32 C adalah asumsi; cantumkan sumber agronomi (literatur/dosen) di laporan.

In [ ]:
def tentukan_kelas(row):
    soil, temp = row["soil_pct"], row["Temperature"]
    if soil > 60:
        return 0                      # Tanah Cukup
    elif soil >= 40:                  # 40 - 60 %
        return 1 if temp >= 30 else 0 # Siram Ringan jika panas
    else:                             # < 40 %
        return 2 if temp >= 32 else 1 # Wajib Siram jika sangat panas

df["Pump_Class"] = df.apply(tentukan_kelas, axis=1)
df["Pump_Class"].value_counts().sort_index()

,count
Pump_Class,
0,1621
1,1048
2,331


## 4. Pilih fitur dan bagi data
`Pump Data` (label lama) **tidak** dipakai sebagai fitur. `Air Humidity` tidak dipakai karena tidak berpengaruh pada label. **Urutan FITUR harus sama persis dengan array di ESP32.**

In [ ]:
FITUR = ["soil_pct", "Temperature"]
X, y = df[FITUR], df["Pump_Class"]
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=1)

## 5. Latih decision tree
`max_depth=4`: kedalaman 3 terlalu dangkal (sekitar 11 data di dekat batas 40% salah).

In [ ]:
model = DecisionTreeClassifier(max_depth=4, random_state=1).fit(Xtr, ytr)

## 6. Evaluasi

In [ ]:
NAMA_KELAS = ["Tanah Cukup", "Siram Ringan", "Wajib Siram"]
pred = model.predict(Xte)
print(classification_report(yte, pred, target_names=NAMA_KELAS, digits=3))
print("Confusion matrix:\n", confusion_matrix(yte, pred))
cv = cross_val_score(DecisionTreeClassifier(max_depth=4, random_state=1), X, y, cv=5)
print("Akurasi validasi silang 5-lipatan: %.4f" % cv.mean())
print("Pentingnya fitur:", dict(zip(FITUR, model.feature_importances_.round(3))))
print(export_text(model, feature_names=FITUR))

              precision    recall  f1-score   support

 Tanah Cukup      1.000     0.997     0.998       324
Siram Ringan      0.995     1.000     0.998       210
 Wajib Siram      1.000     1.000     1.000        66

    accuracy                          0.998       600
   macro avg      0.998     0.999     0.999       600
weighted avg      0.998     0.998     0.998       600

Confusion matrix:
 [[323   1   0]
 [  0 210   0]
 [  0   0  66]]
Akurasi validasi silang 5-lipatan: 0.9987
Pentingnya fitur: {'soil_pct': np.float64(0.614), 'Temperature': np.float64(0.386)}
|--- soil_pct <= 43.19
|   |--- Temperature <= 32.00
|   |   |--- soil_pct <= 40.22
|   |   |   |--- class: 1
|   |   |--- soil_pct >  40.22
|   |   |   |--- Temperature <= 29.43
|   |   |   |   |--- class: 0
|   |   |   |--- Temperature >  29.43
|   |   |   |   |--- class: 1
|   |--- Temperature >  32.00
|   |   |--- soil_pct <= 39.96
|   |   |   |--- class: 2
|   |   |--- soil_pct >  39.96
|   |   |   |--- class: 1
|--- so

## 7. Ekspor ke C++ untuk ESP32
Hasilnya `model_3kelas.h` (sekitar 2 KB, tanpa library tambahan). Tambahkan ke proyek Velxio/Arduino bersama `sketch_ai_3kelas.ino`.

In [ ]:
with open("model_3kelas.h", "w") as f:
    f.write(port(model, classname="PenyiramanModel3Kelas"))
print("model_3kelas.h dibuat")

model_3kelas.h dibuat


## KODE PROGRAM


In [ ]:
#define BLYNK_TEMPLATE_ID   "TMPL6z2uH3wrp"
#define BLYNK_TEMPLATE_NAME "IOT"
#define BLYNK_AUTH_TOKEN    "THjb1RpmtbOApx-uoDm9tQB8xYZWP0hp"

#define BLYNK_PRINT Serial

// Aktifkan Pengujian Model AI (Set 1 untuk aktif, 0 untuk nonaktif)
#define TEST_MODE 1

#include <WiFi.h>
#include <WiFiClient.h>
#include <BlynkSimpleEsp32.h>
#include <WiFiManager.h>
#include <Wire.h>
#include <LiquidCrystal_I2C.h>
#include <DHT.h>

// 1. Inklusi Header Model Machine Learning (3 Kelas)
#include "model_3kelas.h"

#define DHTPIN 4
#define DHTTYPE DHT22

#define PIN_SOIL 34
#define PIN_BUTTON 18

// ============================================================================
// KALIBRASI SOIL MOISTURE TERKALIBRASI
// ============================================================================
const int RAW_DRY = 2520;
const int RAW_WET = 900;

float calibratePercent(float value) {
  const float measured[] = {0, 28, 45, 50, 73, 92, 98, 100};
  const float target[]   = {0, 25, 45, 50, 70, 88.68f, 93.40f, 100};
  value = constrain(value, 0.0f, 100.0f);

  for (int i = 0; i < 7; ++i) {
    if (value <= measured[i + 1]) {
      const float fraction = (value - measured[i]) /
                             (measured[i + 1] - measured[i]);
      return constrain(target[i] + fraction *
                       (target[i + 1] - target[i]), 0.0f, 100.0f);
    }
  }
  return 100.0f;
}

float hitungPersentaseTanahTerkalibrasi(int raw) {
  if (RAW_DRY == RAW_WET) return 0.0f;

  const int RAW_MID = RAW_DRY - (56 * (RAW_DRY - RAW_WET) / 100);
  float moisture;
  if (raw >= RAW_MID) {
    moisture = 50.0f * (RAW_DRY - raw) / (float)(RAW_DRY - RAW_MID);
  } else {
    moisture = 50.0f + 50.0f * (RAW_MID - raw) /
               (float)(RAW_MID - RAW_WET);
  }
  moisture = constrain(moisture, 0.0f, 100.0f);
  return calibratePercent(moisture);
}

LiquidCrystal_I2C lcd(0x27, 16, 2);
DHT dht(DHTPIN, DHTTYPE);
BlynkTimer timer;
WiFiManager wifiManager;
Eloquent::ML::Port::PenyiramanModel3Kelas clf;

int currentScreen = 0;
const int TOTAL_SCREENS = 4;

bool lastButtonState = HIGH;
unsigned long lastDebounceTime = 0;

float suhu = 0.0;
float kelembapanUdara = 0.0;
float persentaseTanah = 0.0f;

String statusRekomendasi = "Tdk prlu siram";
String warnaRekomendasi  = "#2E7D32";

// --- VARIABEL PELACAK STATUS WIFI & RECONNECT ---
bool wasConnected = false;
String lastSSID = "";
unsigned long lastReconnectAttempt = 0;

// ============================================================================
// 2. DEFINISI STRUKTUR & FUNGSI UJI MODEL
// ============================================================================
struct Kasus { float tanah; float suhu; int harapan; };
const Kasus uji[] = {
  {20,35,2},{20,25,1},{39,32.5,2},{39,31.5,1},{50,33,1},{50,25,0},
  {59,30.5,1},{59,29.5,0},{61,40,0},{80,25,0},{0,40,2},{100,20,0},
  {41,29.5,0},{40,29.5,0},{60,30.5,1}
};

void jalankanUjiModel() {
  int n = sizeof(uji) / sizeof(uji[0]), lulus = 0;
  for (int i = 0; i < n; i++) {
    float x[2] = { uji[i].tanah, uji[i].suhu };
    int h = clf.predict(x);
    bool ok = (h == uji[i].harapan);
    if (ok) lulus++;
  }
}

// ============================================================================
// 3. FUNGSI SENSOR & LOGIKA UTAMA
// ============================================================================
void bacaSemuaSensor() {
  kelembapanUdara = dht.readHumidity();
  suhu = dht.readTemperature();

  uint32_t totalSoil = 0;
  for (int i = 0; i < 16; ++i) {
    totalSoil += analogRead(PIN_SOIL);
    delay(2);
  }
  const int rawSoil = totalSoil / 16;
  persentaseTanah = hitungPersentaseTanahTerkalibrasi(rawSoil);

  if (!isnan(suhu)) {
    float x_input[2] = { persentaseTanah, suhu };
    int hasilPrediksi = clf.predict(x_input);

    if (hasilPrediksi == 2) {
      statusRekomendasi = "Wajib Siram!";
      warnaRekomendasi  = "#D32F2F";
    } else if (hasilPrediksi == 1) {
      statusRekomendasi = "Siram Ringan";
      warnaRekomendasi  = "#F9A825";
    } else {
      statusRekomendasi = "Tdk prlu siram";
      warnaRekomendasi  = "#2E7D32";
    }
  }
}

void kirimDataKeBlynkDanUpdateLCD() {
  bacaSemuaSensor();

  if (Blynk.connected() && WiFi.status() == WL_CONNECTED) {
    if (!isnan(suhu)) Blynk.virtualWrite(V0, suhu);
    if (!isnan(kelembapanUdara)) Blynk.virtualWrite(V1, kelembapanUdara);
    Blynk.virtualWrite(V2, persentaseTanah);
    Blynk.setProperty(V4, "color", warnaRekomendasi);
    Blynk.virtualWrite(V4, statusRekomendasi);
  }

  tampilkanLayar();
}

void tampilkanLayar() {
  char baris1[17] = "";
  char baris2[17] = "";

  switch (currentScreen) {
    case 0:
      snprintf(baris1, sizeof(baris1), "Rekomendasi AI :");
      snprintf(baris2, sizeof(baris2), "> %-14s", statusRekomendasi.c_str());

      lcd.setCursor(0, 0);
      lcd.print(baris1);
      lcd.setCursor(0, 1);
      lcd.print(baris2);
      break;

    case 1:
      if (isnan(suhu)) {
        snprintf(baris1, sizeof(baris1), "Suhu  : Error   ");
      } else {
        snprintf(baris1, sizeof(baris1), "Suhu  : %4.1f C ", suhu);
      }

      if (isnan(kelembapanUdara)) {
        snprintf(baris2, sizeof(baris2), "Lembap: Error   ");
      } else {
        snprintf(baris2, sizeof(baris2), "Lembap: %4.1f %% ", kelembapanUdara);
      }

      lcd.setCursor(0, 0);
      lcd.print(baris1);
      lcd.setCursor(0, 1);
      lcd.print(baris2);
      break;

    case 2:
      snprintf(baris1, sizeof(baris1), "Tanah:%5.1f%%    ", persentaseTanah);

      if (persentaseTanah < 40) {
        snprintf(baris2, sizeof(baris2), "Status: Kering  ");
      } else if (persentaseTanah <= 70) {
        snprintf(baris2, sizeof(baris2), "Status: Optimal ");
      } else {
        snprintf(baris2, sizeof(baris2), "Status: Basah   ");
      }

      lcd.setCursor(0, 0);
      lcd.print(baris1);
      lcd.setCursor(0, 1);
      lcd.print(baris2);
      break;

    case 3:
      if (Blynk.connected() && WiFi.status() == WL_CONNECTED) {
        snprintf(baris1, sizeof(baris1), "Blynk: Connected");
      } else {
        snprintf(baris1, sizeof(baris1), "Blynk: Offline  ");
      }

      if (WiFi.status() == WL_CONNECTED) {
        snprintf(baris2, sizeof(baris2), "WiFi :%-10s", lastSSID.c_str());
      } else {
        if (lastSSID != "") {
          snprintf(baris2, sizeof(baris2), "X %-14s", lastSSID.c_str());
        } else {
          snprintf(baris2, sizeof(baris2), "WiFi :Setup AP  ");
        }
      }

      lcd.setCursor(0, 0);
      lcd.print(baris1);
      lcd.setCursor(0, 1);
      lcd.print(baris2);
      break;
  }

  // TERAKHIR: Kirim hasil rakitan baris1 dan baris2 ke Virtual Pin Blynk
  kirimTeksLcdKeBlynk(String(baris1), String(baris2));
}

// ============================================================================
// 4. SETUP UTAMA
// ============================================================================
void setup() {
  Serial.begin(115200);

  #if TEST_MODE
    jalankanUjiModel();
  #endif

  pinMode(PIN_BUTTON, INPUT_PULLUP);
  pinMode(PIN_SOIL, INPUT);
  analogSetPinAttenuation(PIN_SOIL, ADC_11db);
  analogReadResolution(12);

  Wire.begin(21, 23);
  lcd.init();
  lcd.backlight();
  dht.begin();

  lcd.setCursor(0, 0);
  lcd.print("Memulai Sistem..");
  delay(1000);

  // Matikan auto-reconnect bawaan agar tidak merusak sinyal AP
  WiFi.setAutoReconnect(false);

  wifiManager.setConfigPortalBlocking(false);
  wifiManager.autoConnect("SmartPlant_Setup");
  if (lastSSID == "") lastSSID = wifiManager.getWiFiSSID();
  Blynk.config(BLYNK_AUTH_TOKEN);
  timer.setInterval(2000L, kirimDataKeBlynkDanUpdateLCD);

  if (WiFi.status() == WL_CONNECTED) {
    wasConnected = true;
    lastSSID = WiFi.SSID();
  }

  bacaSemuaSensor();
  lcd.clear();
  tampilkanLayar();
}

void loop() {
  wifiManager.process();

  // --- LOGIKA PENDETEKSI & RECONNECT WIFI ---
  if (WiFi.status() == WL_CONNECTED) {
    if (!wasConnected) {
      wasConnected = true;
      lastSSID = WiFi.SSID();
      Blynk.connect();
      wifiManager.stopConfigPortal();
      tampilkanLayar();
    }
    Blynk.run();
  } else {
    if (wasConnected) {
      wasConnected = false;
      Blynk.disconnect();
      Serial.println("Koneksi terputus dari: " + lastSSID);

      // Paksa ESP32 memancarkan AP sekaligus mencari WiFi
      WiFi.mode(WIFI_AP_STA);
      wifiManager.startConfigPortal("SmartPlant_Setup");
      tampilkanLayar();
    }

    // --- FITUR AUTO-RECONNECT HALUS ---
    if (millis() - lastReconnectAttempt > 10000) {
      lastReconnectAttempt = millis();
      if (lastSSID != "") {
        Serial.println("Mencoba menyambung kembali secara halus ke " + lastSSID + "...");
        WiFi.begin();
      }
    }
  }

  timer.run();

  // =========================================================
  // LOGIKA PUSHBUTTON (Pastikan hanya ada SATU blok ini saja)
  // =========================================================
  int reading = digitalRead(PIN_BUTTON);
  if (reading == LOW && lastButtonState == HIGH && (millis() - lastDebounceTime > 50)) {
    lastDebounceTime = millis();
    currentScreen = (currentScreen + 1) % TOTAL_SCREENS;

    // -- TRIK ANTI BERANTAKAN LCD --
    lcd.init();      // Inisialisasi ulang LCD
    lcd.backlight(); // Nyalakan ulang lampu belakang
    lcd.clear();     // Bersihkan layar

    tampilkanLayar();
  }
  lastButtonState = reading;
}

// Fungsi ini otomatis berjalan sesaat ketika tombol V5 di Blynk ditekan/berubah
// ============================================================================
// 5. LOGIKA TOMBOL BLYNK V6 (SIMULASI PUSHBUTTON FISIK)
// ============================================================================
BLYNK_WRITE(V6) {
  int statusTombol = param.asInt();

  // Hanya bereaksi ketika tombol ditekan (nilai 1)
  if (statusTombol == 1) {
    Serial.println("Tombol Blynk V6 ditekan: Ganti Layar LCD");

    // Logika ganti layar yang sama persis dengan tombol fisik
    currentScreen = (currentScreen + 1) % TOTAL_SCREENS;
    lcd.clear();
    tampilkanLayar();
  }
}

// ============================================================================
// 6. FUNGSI MIRROR LCD KE BLYNK
// ============================================================================
void kirimTeksLcdKeBlynk(String baris1, String baris2) {
  if (Blynk.connected()) {
    Blynk.virtualWrite(V10, baris1); // Virtual Pin V10 -> Baris 1 LCD Widget
    Blynk.virtualWrite(V11, baris2); // Virtual Pin V11 -> Baris 2 LCD Widget
  }
}